In [161]:
# Importing the libraries that will be needed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3 as sql

In [162]:
conn = sql.connect("download.db")

In [163]:
# Reading members table
members = pd.read_sql('SELECT * FROM MEMBERS', conn)
members.head()

,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05
1,1002,Fares,Saleh,9.0,Maadi,Active,None
2,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23
3,1004,Fares,Wahba,7.0,Maadi,inactive,2024-10-09
4,1005,Youssef,Halim,9.0,Maadi,Active,2024-05-05


In [164]:
# Reading books table
books = pd.read_sql('SELECT * FROM BOOKS', conn)
books.head()

,book_id,title,author
0,501,The Silver Kite,Amina Darwish
1,502,Desert Compass,Amina Darwish
2,503,The Lantern Maker,Adel Roushdy
3,504,Rooftop Astronomers,Adel Roushdy
4,505,Letters to the Nile,Aya Hafez


In [165]:
# Reading checkouts table
checkouts = pd.read_sql('SELECT * FROM CHECKOUTS', conn)
checkouts.head()

,checkout_id,member_id,book_id,checkout_date,return_date
0,9263,1047,517,2024-10-21,2024-11-07
1,9340,1072,513,2025-08-24,2025-09-01
2,9231,1053,523,2024-02-04,2024-02-16
3,9129,1032,513,2025-06-21,2025-06-29
4,9370,1079,511,2025-11-11,2025-12-03


In [166]:
# Making a function ro read sql
def sql_query(q):
    return pd.read_sql(q, conn)

In [167]:
# Counting the total checkouts for each member, use left join to show members with zero (0) checkouts
count = """
SELECT M.member_id, M.first_name, last_name, COUNT(C.checkout_id) AS Borrowing_COUNT
FROM MEMBERS M
LEFT JOIN CHECKOUTS C
ON M.member_id = C.member_id
GROUP BY M.member_id
"""
sql_query(count)

,member_id,first_name,last_name,Borrowing_COUNT
0,1001,Salma,Ibrahim,1
1,1002,Fares,Saleh,2
2,1003,Bassel,Hegazy,9
3,1004,Fares,Wahba,0
4,1005,Youssef,Halim,3
...,...,...,...,...
75,1076,Dina,Wahba,7
76,1077,Lina,Rashad,6
77,1078,Habiba,Osman,0
78,1079,Rana,Osman,10


In [168]:
# Showing the titles of books whose aauthor name starts with M 
like = """
SELECT title
FROM BOOKS
WHERE author like "M%"
"""
print("The titles of the books whose author name starts with M")
sql_query(like)

The titles of the books whose author name starts with M


,title
0,Storms and Sailboats
1,The Clockwork Orchard
2,Notes from the Delta
3,The Glass Beehive


In [169]:
# Showing the most 5 borrowed books with counting the checkouts
popular_books = """
SELECT B.book_id, B.title, COUNT(c.checkout_id) AS COUNT
FROM BOOKS B
JOIN CHECKOUTS C
ON B.book_id = C.book_id
GROUP BY B.book_id 
ORDER BY COUNT DESC LIMIT 5
"""
sql_query(popular_books)

,book_id,title,COUNT
0,501,The Silver Kite,57
1,507,Fossils and Fireflies,55
2,513,Circuits for Beginners,46
3,519,Kites Over Cairo,38
4,525,Storms and Sailboats,25


In [170]:
# Showing the most 10 active members with counting their checkouts
active_readers = """
SELECT M.member_id, M.first_name, last_name, COUNT(C.checkout_id) AS Borrowing_Count
FROM MEMBERS M
JOIN CHECKOUTS C
ON M.member_id = C.member_id
GROUP BY M.member_id
ORDER BY Borrowing_Count DESC LIMIT 10
"""
sql_query(active_readers)

,member_id,first_name,last_name,Borrowing_Count
0,1034,Aya,Wahba,25
1,1044,Sherif,Saleh,21
2,1008,Ziad,Saleh,19
3,1027,Mostafa,Fouad,18
4,1010,Nour,Nabil,18
5,1065,Adam,Fahmy,17
6,1024,Youssef,Hegazy,17
7,1018,Ahmed,Shafik,17
8,1047,Sara,Rashad,16
9,1030,Reem,Osman,16


In [171]:
# Showing Maadi activity'look from newest to oldest through past the ten most recent
maadi = """
SELECT M.member_id, M.first_name, M.neighborhood, C.checkout_date
FROM MEMBERS M
JOIN CHECKOUTS C
ON M.member_id = C.member_id
WHERE M.neighborhood = "Maadi"
ORDER BY C.checkout_date DESC
LIMIT 10 OFFSET 10
"""
print("Neighborhood is Maadi")
sql_query(maadi)

Neighborhood is Maadi


,member_id,first_name,neighborhood,checkout_date
0,1003,Bassel,Maadi,2025-09-04
1,1017,Adam,Maadi,2025-08-25
2,1008,Ziad,Maadi,2025-08-23
3,1003,Bassel,Maadi,2025-08-21
4,1018,Ahmed,Maadi,2025-08-19
5,1015,Hamza,Maadi,2025-08-08
6,1018,Ahmed,Maadi,2025-08-04
7,1013,Ziad,Maadi,2025-07-27
8,1003,Bassel,Maadi,2025-07-22
9,1009,Hassan,Maadi,2025-07-22


In [172]:
# Combine members table with checkout table, count the number of borrowed books for each member by using transform function
combined_sql = pd.merge(members, checkouts, on = "member_id", how = "left")
combined_sql["borrowed_books"] = combined_sql.groupby("member_id")["checkout_id"].transform("count")
combined_sql.head()

,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date,checkout_id,book_id,checkout_date,return_date,borrowed_books
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05,9025.0,525.0,2024-10-24,2024-11-07,1
1,1002,Fares,Saleh,9.0,Maadi,Active,None,9013.0,501.0,2024-02-16,2024-02-29,2
2,1002,Fares,Saleh,9.0,Maadi,Active,None,9095.0,507.0,2024-06-25,2024-07-07,2
3,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9051.0,513.0,2025-07-22,2025-08-18,9
4,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9068.0,501.0,2025-11-14,2025-12-03,9


In [173]:
# Showing the number of rows and columns in the combined_sql DataFrame
combined_sql.shape

(409, 12)

In [174]:
# Reading the json file
json = pd.read_json("download.json")
json.head()

,book_id,genre,pages,publication_year,publisher
0,501,Adventure,128,2017.0,Nile Press
1,502,Adventure,109,2018.0,Delta House
2,503,Historical,259,NaN,Nile Press
3,504,Science,319,2009.0,Cairo Young Readers
4,505,Historical,216,2024.0,Oasis Books


In [175]:
# Showing the number of rows and columns in the json
json.shape

(32, 5)

In [176]:
# Combining the combined_sql with the json by book_id
combined_json_sql = pd.merge(combined_sql, json, on = "book_id", how = "left")
combined_json_sql.head()

,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date,checkout_id,book_id,checkout_date,return_date,borrowed_books,genre,pages,publication_year,publisher
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05,9025.0,525.0,2024-10-24,2024-11-07,1,Adventure,297.0,2015.0,Oasis Books
1,1002,Fares,Saleh,9.0,Maadi,Active,None,9013.0,501.0,2024-02-16,2024-02-29,2,Adventure,128.0,2017.0,Nile Press
2,1002,Fares,Saleh,9.0,Maadi,Active,None,9095.0,507.0,2024-06-25,2024-07-07,2,Science,160.0,2024.0,Nile Press
3,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9051.0,513.0,2025-07-22,2025-08-18,9,Science,294.0,2021.0,Oasis Books
4,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9068.0,501.0,2025-11-14,2025-12-03,9,Adventure,128.0,2017.0,Nile Press


In [177]:
# Showing the number of rows and columns in the combined_json_sql
combined_json_sql.shape

(409, 16)

In [178]:
# Reading the html file and selecting the first table in the html file
html = pd.read_html("download.html")
html_table = html[0]
html_table.head()

,Member ID,Book ID,Checkout Date
0,1026,522,2025-07-11
1,1049,520,2025-07-11
2,1062,525,2025-07-05
3,1065,520,2025-07-07
4,1104,515,2025-07-07


In [179]:
# Showing the number of rows and columns in the html_table
html_table.shape

(26, 3)

In [180]:
# Changing the column names in the html_table by making them lower case and replacing spaces with underscores
html_table.columns = html_table.columns.str.lower().str.replace(" ", "_")
html_table.head()

,member_id,book_id,checkout_date
0,1026,522,2025-07-11
1,1049,520,2025-07-11
2,1062,525,2025-07-05
3,1065,520,2025-07-07
4,1104,515,2025-07-07


In [181]:
# Combining the combined_json_sql with the html_table
combined_data = pd.concat([combined_json_sql, html_table], ignore_index=True)
combined_data.head()

,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date,checkout_id,book_id,checkout_date,return_date,borrowed_books,genre,pages,publication_year,publisher
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05,9025.0,525.0,2024-10-24,2024-11-07,1.0,Adventure,297.0,2015.0,Oasis Books
1,1002,Fares,Saleh,9.0,Maadi,Active,None,9013.0,501.0,2024-02-16,2024-02-29,2.0,Adventure,128.0,2017.0,Nile Press
2,1002,Fares,Saleh,9.0,Maadi,Active,None,9095.0,507.0,2024-06-25,2024-07-07,2.0,Science,160.0,2024.0,Nile Press
3,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9051.0,513.0,2025-07-22,2025-08-18,9.0,Science,294.0,2021.0,Oasis Books
4,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23,9068.0,501.0,2025-11-14,2025-12-03,9.0,Adventure,128.0,2017.0,Nile Press


In [182]:
# Showing the number of rows and columns in the combined_data
combined_data.shape

(435, 16)

In [183]:
combined_data.columns

Index(['member_id', 'first_name', 'last_name', 'grade', 'neighborhood',
       'membership_status', 'join_date', 'checkout_id', 'book_id',
       'checkout_date', 'return_date', 'borrowed_books', 'genre', 'pages',
       'publication_year', 'publisher'],
      dtype='object')

In [184]:
# Saving the combined_data to a csv file
combined_data.to_csv("task1_combined_data.csv", index=False)